# 4장 2강 · 이미지 군집화와 결과 해석

실습문제

사진 자료실에 들어온 반려동물 사진을 묶어 검토하려고 합니다. 대표 사진 한 장으로 폴더를 설명해도 될까요? 기존 폴더와 잘 어울리지 않는 새 사진은 어떤 순서로 열어 봐야 할까요? 실제 사진의 거리와 품종 분포를 근거로 답해 봅시다.

## 실습 목표와 데이터

| 구간 | 판단할 내용 | 제출할 근거 |
|---|---|---|
| 필수 1 | K-Means가 사진을 어느 중심에 배정했는가? | 파일별 소속, 직접 계산한 거리와 중심, 거리 제곱합 |
| 필수 2 | 묶음을 설명하는 사진과 사람이 더 살펴볼 사진은 각각 무엇인가? | 대표·구성 사진, 두 중심까지의 거리 차이, K=2와 K=3의 대응표 |
| 과제 1 | 사진이 늘어나면 어떤 수의 폴더를 만들고 무엇부터 검토할까? | K 후보별 지표, 5개 폴더의 구성, 새 사진 검토 목록 |

**준비물**  강의용 `oxford_dataset.zip`을 압축 상태로 노트북과 같은 폴더에 둡니다. 다른 위치라면 첫 코드 셀의 `paths`를 수정하세요. ZIP 안의 `val.csv`, `test.csv`와 거기에 기록된 이미지 경로를 읽습니다. Python 3, Jupyter, PyTorch, torchvision, scikit-learn, NumPy, pandas, Pillow를 사용합니다. 검증한 모델 환경은 CPU용 PyTorch 2.6.0, torchvision 0.21.0, scikit-learn 1.8.0입니다. ResNet18 가중치는 처음 사용할 때 다운로드하며 이후에는 로컬 캐시를 사용합니다.

**실행 순서**  공통 준비 두 셀을 실행한 뒤 각 구간 안에서는 번호순으로 진행하세요. 필수 2와 과제 1은 공통 준비 직후에도 시작할 수 있습니다. 품종명은 표본을 고르고 결과를 해석할 때 사용합니다. 군집 모델에 넣는 입력은 사진에서 얻은 숫자뿐입니다.

### 공통 준비 ① · 같은 사진을 두 번 세지 않도록 나누기

| 용도 | 사진 수 | 선택 방법 |
|---|---:|---|
| 기본 실습 | 12 | 기존 실습의 비글·퍼그·페르시안 각 4장. 파일명을 고정합니다. |
| 확장 실험의 군집 학습 | 120 | 비글·퍼그·복서·페르시안·버만 각 24장. 이 사진으로 중심을 구합니다. |
| 확장 실험의 새 사진 | 40 | 위 다섯 품종 각 8장. 중심을 정한 뒤 처음 배정합니다. |

기본 사진은 배포본의 val 목록에서 읽습니다. 확장 실험은 test 목록에서 품종별 32장을 중복 없이 뽑아 24장과 8장으로 나눕니다. 여기서 ‘학습’과 ‘새 사진’은 이번 군집 실험 안에서의 역할입니다. `sort_index()`로 후보 순서를 고정하고 난수 시드 42를 사용합니다. 이미지 ID 간 중복도 검사합니다.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display
import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from threadpoolctl import threadpool_limits

paths = [Path("oxford_dataset.zip")]
data_path = next((p for p in paths if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("강의용 Oxford ZIP을 노트북 옆에 두거나 paths를 수정하세요.")

demo_ids = ["beagle_110", "beagle_111", "beagle_117", "beagle_126",
            "pug_100", "pug_103", "pug_105", "pug_116",
            "Persian_100", "Persian_125", "Persian_135", "Persian_138"]
breeds = ["beagle", "pug", "boxer", "Persian", "Birman"]
fit_ids, new_ids = [], []
rng = np.random.default_rng(42)
with ZipFile(data_path) as archive:
    val = pd.read_csv(archive.open("val.csv")).set_index("image_id")
    test = pd.read_csv(archive.open("test.csv")).set_index("image_id")
    available = test.loc[test.image_path.isin(archive.namelist())].sort_index()
    for breed in breeds:
        pool = available.index[available.class_name == breed]
        chosen = rng.choice(pool, 32, replace=False).tolist()
        fit_ids.extend(chosen[:24])
        new_ids.extend(chosen[24:])
    all_ids = demo_ids + fit_ids + new_ids
    assert len(set(all_ids)) == len(all_ids)
    meta = pd.concat([val.loc[demo_ids], test.loc[fit_ids + new_ids]])
    photos = {}
    for key in all_ids:
        with Image.open(BytesIO(archive.read(meta.at[key,"image_path"]))) as im:
            photos[key] = im.convert("RGB")
print("기본 / 확장 학습 / 새 사진", len(demo_ids), len(fit_ids), len(new_ids))

def photo_grid(keys):
    cols = min(4, len(keys))
    sheet = Image.new("RGB", (cols*162, ((len(keys)+cols-1)//cols)*145), "white")
    draw = ImageDraw.Draw(sheet)
    for i,key in enumerate(keys):
        im = ImageOps.contain(photos[key], (158,117))
        x, y = (i%cols)*162, (i//cols)*145
        sheet.paste(im, (x+(158-im.width)//2,y))
        draw.text((x+3,y+120),key,fill="black")
    return sheet

기본 / 확장 학습 / 새 사진 12 120 40


### 공통 준비 ② · 사진을 같은 좌표계의 특징으로 나타내기

원본을 RGB로 통일하고 사전학습 가중치의 전처리를 적용합니다. 짧은 변을 256으로 조정하고 가운데 224×224를 자른 다음, 픽셀을 텐서로 바꾸고 채널별로 표준화합니다. `eval()`은 모델을 평가 모드로 돌리고 `inference_mode()`는 추론 중 기울기 기록을 생략합니다. 마지막 분류층을 `Identity()`로 바꾸면 사진별 512개 특징이 그대로 나옵니다. 여기서 ResNet18은 특징 추출기로 사용하고, K-Means는 다음 문항에서 사진으로 새로 학습합니다.

`torch.stack`은 여러 사진 앞에 배치 축을 추가합니다. 최대 16장씩 처리해 메모리 사용량을 줄입니다. `features`의 행 이름은 파일 ID입니다. `matrix(keys)`는 지정한 ID 순서대로 숫자 행만 가져옵니다. 모든 실험은 이렇게 얻은 원래 임베딩의 유클리드 거리를 사용합니다.

In [2]:
torch.set_num_threads(2)
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
encoder = resnet18(weights=weights)
encoder.fc = nn.Identity()
encoder.eval()
parts = []
with torch.inference_mode():
    for start in range(0, len(all_ids), 16):
        keys = all_ids[start:start+16]
        batch = torch.stack([preprocess(photos[key]) for key in keys])
        parts.append(encoder(batch).numpy())
features = pd.DataFrame(np.concatenate(parts), index=all_ids)
def matrix(keys):
    return features.loc[keys].to_numpy(dtype=np.float64)
X_demo, X_fit, X_new = [matrix(keys) for keys in (demo_ids,fit_ids,new_ids)]
print("임베딩 전체 / 기본 / 확장 학습 / 새 사진",features.shape,X_demo.shape,X_fit.shape,X_new.shape)

def fit_kmeans(X, k):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    with threadpool_limits(limits=1):
        model.fit(X)
    return model

임베딩 전체 / 기본 / 확장 학습 / 새 사진 (172, 512) (12, 512) (120, 512) (40, 512)


## 필수 1 · 사진 12장을 두 군집으로 묶기

군집 모델이 실제로 받은 입력부터 확인합니다. 사진 한 장을 골라 거리, 소속, 중심 갱신을 손으로 따라간 다음 12장 전체에서 검산하세요.

### 1-1. 사진 ID와 임베딩 한 행을 연결하기

기본 표본의 첫 사진에 대해 원본 크기, 전처리 텐서의 모양, 임베딩 앞 다섯 값을 출력하세요. ID로 다시 꺼낸 행과 첫 행이 같은지, 기본 표본에 결측·무한값이 없는지도 확인하세요. 품종 분류와 이번 군집화에 쓰이는 정보의 차이를 적으세요.

**풀이 단서**  
Pillow의 size는 가로부터, PyTorch 이미지 텐서는 채널부터 읽습니다. np.array_equal은 두 배열을 원소별로 비교합니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# key = demo_ids[0]
# photos[key], preprocess, X_demo, matrix([key])를 이용하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 1-2. K=2를 학습하고 모든 사진에 소속 붙이기

기본 표본에 K=2, random_state=42, n_init=10으로 K-Means를 학습하세요. 파일별 군집 번호와 군집별 사진 수, 선택된 실행의 반복 횟수를 확인하고 n_init과 반복 횟수의 차이를 설명하세요.

**풀이 단서**  
공통 함수 fit_kmeans(X, k)는 .fit()까지 끝난 모델을 반환합니다. labels_의 n번째 원소를 n번째 사진에 붙이세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# model2 = fit_kmeans(...)
# base의 인덱스는 demo_ids, cluster 열의 값은 학습 결과입니다.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 1-3. 한 사진의 거리와 한 중심의 평균을 직접 검산하기

첫 사진에서 두 중심까지의 거리를 NumPy로 직접 구하세요. 최소 거리의 번호와 모델 소속을 비교한 뒤 12장 전체에서도 같은지 검사하세요. 첫 사진이 속한 군집의 평균 벡터를 직접 구해 모델 중심과 대조하세요.

**풀이 단서**  
사진에서 중심을 빼면 (2, 512)입니다. sum(axis=1)은 각 중심의 512개 차이를 더하고, mean(axis=0)은 소속 사진들을 모아 특징별 평균을 냅니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# dist2 = model2.transform(X_demo)
# 직접 계산은 X_demo[0]과 model2.cluster_centers_에서 시작하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 1-4. 거리 제곱합과 품종 교차표는 무엇을 재는가?

각 사진에서 자기 중심까지의 거리를 제곱해 모두 더하고 모델의 inertia_와 비교하세요. 학습을 마친 다음 ID로 품종을 붙여 군집×품종 교차표를 만드세요. 두 결과가 각각 답하는 질문을 쓰세요.

**풀이 단서**  
거리표에 min(axis=1)을 적용하면 사진별 최솟값이 남습니다. crosstab의 한 칸은 행과 열의 조건을 동시에 만족한 사진 수입니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# own_dist = ...
# 품종은 meta.loc[base.index, "class_name"]으로 정렬하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

## 필수 2 · 대표 사진과 추가 검토 사진 고르기

각 군집의 표지 사진 한 장을 고른 다음, 그 표지가 놓치는 사진을 찾아보세요. 같은 사진 12장으로 K를 바꿔 어떤 묶음이 나뉘는지도 추적합니다.

### 2-1. 군집 중심과 가장 가까운 실제 사진 고르기

기본 12장에 K=2를 학습하고 각 사진의 소속 중심까지 거리를 구하세요. 군집마다 대표 한 장을 뽑아 ID·군집·거리와 두 사진을 제시하세요. 중심과 대표 사진이 어떻게 다른지 설명하세요.

**풀이 단서**  
각 군집 안에서 distance가 가장 작은 행을 고릅니다. 거리 동점일 때는 image_id 오름차순을 사용하세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# m = fit_kmeans(X_demo, 2)
# ranked를 cluster, distance, image_id 순서로 정렬하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 2-2. 대표와 다른 구성 사진 찾기

두 군집의 전체 사진을 거리 오름차순으로 표시하세요. 각 군집에서 대표와 나머지 사진의 공통점 및 차이점을 하나씩 찾아 파일 ID와 함께 기록하세요. 화면에서 관찰한 사실과 모델 판단 원인에 대한 추측을 구분하세요.

**풀이 단서**  
groupby 후 기존 거리순을 유지하세요. 첫 사진이 대표이고 마지막은 그 군집 중심에서 가장 먼 사진입니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# for cid, group in ranked.groupby("cluster"):
#     거리 표와 photo_grid(group.image_id.tolist())를 출력하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 2-3. 중심에서 먼 사진과 배정이 아슬아슬한 사진 구별하기

각 사진에서 가까운 중심 거리(own), 두 번째 중심 거리(other), 두 거리 차이(gap)를 구하세요. gap이 작은 3장과 own이 큰 3장을 각각 제시하고 두 목록의 첫 사진이 다른 이유를 설명하세요.

**풀이 단서**  
거리를 행별 정렬한 뒤 두 번째 열에서 첫 번째 열을 뺍니다. gap은 이 실습에서 쓰는 검토 기준이며 품종 확률을 뜻하지 않습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# ordered_d = np.sort(d, axis=1)
# border에 own, other, gap을 만들고 두 가지 기준으로 정렬하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 2-4. K=2에서 K=3으로 바뀐 묶음 추적하기

같은 12장의 임베딩으로 K=3을 새로 학습하세요. K2×K3 소속 교차표와 K3×품종 교차표를 만든 뒤 K=3의 대표 사진과 거리를 다시 구하세요. 어떤 기존 묶음이 어떻게 나뉘었는지 파일과 수치로 설명하세요.

**풀이 단서**  
두 모델의 번호는 독립적입니다. 같은 파일의 옛 번호와 새 번호를 한 행에 놓고 crosstab으로 세세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# m3 = fit_kmeans(X_demo, 3)
# 대표는 m3의 중심 거리로 새로 선택하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

## 과제 1 · 사진이 늘어났을 때 군집 수와 검토 순서 결정하기

사진 담당자는 120장으로 폴더를 만든 후 새 사진 40장을 기존 폴더에 넣고 싶습니다. 먼저 품종을 가리고 사진 간 거리만으로 K 후보를 비교하세요. 이어서 **운영팀이 품종별 후보를 살펴볼 폴더 5개를 요청했다**는 조건으로 K=5를 검토합니다. 새 사진의 실제 품종은 배정이 끝난 뒤에만 확인하세요.

### 3-1. 확장 학습 사진 120장으로 K 후보 비교하기

학습 사진만 사용해 K=2, 3, 4, 5, 6을 각각 학습하세요. 같은 표에 거리 제곱합, 평균 실루엣, 가장 작은·큰 군집 장수를 적으세요. K=3에서 첫 사진의 실루엣을 직접 계산하고 어떤 기준에서 K를 택할지 설명하세요.

**풀이 단서**  
실루엣의 a는 자신을 뺀 같은 군집 사진까지의 평균 거리, b는 다른 군집별 평균 거리 중 가장 작은 값입니다. 개별 값은 (b−a)/max(a,b)입니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# models = {}  # K별 학습 결과를 보관하세요.
# silhouette_score(X_fit, labels), np.bincount(labels)를 활용하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 3-2. K=5 폴더에 품종 이름을 붙여도 될까?

운영 조건에 따라 K=5 모델을 선택하세요. 학습 사진의 군집×품종 교차표를 만들고, 각 폴더의 최다 품종·해당 장수·폴더 전체 대비 비율을 구하세요. 같은 품종이 둘 이상의 폴더에 나뉜 사례를 한 가지 제시하세요.

**풀이 단서**  
행은 군집입니다. 행 합계가 분모이고 행 최댓값이 분자입니다. 이름표는 학습 사진에서만 결정해 다음 문항에 가져갑니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# chosen_model = models[5]
# mix = pd.crosstab(...)
# dominant = mix.idxmax(axis=1)

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 3-3. 기존 중심을 고정해 새 사진 40장 배정하기

추가 fit 없이 새 사진 40장을 K=5의 기존 중심에 배정하세요. 학습 사진에서 정한 최다 품종으로 후보 이름을 붙이고 실제 품종과 대조하세요. 전체 일치율과 품종별 불일치 방향을 구하세요.

**풀이 단서**  
predict는 기존 중심 중 하나를 고릅니다. 새 자료에 fit_predict를 적용하면 중심과 번호가 달라질 수 있습니다. 정답 열은 배정 후 결합하세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# new_labels = chosen_model.predict(X_new)
# 군집 번호를 dominant로 매핑하고 실제 품종을 ID로 찾아 대조하세요.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

### 3-4. 다섯 장만 먼저 열 수 있을 때 검토 목록 작성하기

새 사진을 gap이 작은 순으로 정렬해 우선 검토 5장의 ID·거리·품종과 사진을 출력하세요. 전체 불일치 중 몇 장을 발견하고 몇 장을 놓쳤는지 구하세요. 담당자에게 ① 관찰 사실 ② 가능한 해석 ③ 운영 방법 ④ 추가 검증을 각각 한 문장으로 제안하세요.

**풀이 단서**  
검토 순서는 gap과 파일 ID만으로 정합니다. 순위 결정이 끝난 후 일치 열을 이용해 검토 목록 안과 밖의 불일치를 각각 셉니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# queue = new.sort_values(["gap","image_id"])
# first = queue.head(5)
# ~는 True와 False를 뒤집습니다.

**나의 해석**  
실제 출력에서 근거를 골라 질문에 답하세요.

✎

## 실습 마무리

1. **어떤 문제가 있었나요?** K=2의 대표 한 장으로 군집을 설명할 때 생기는 문제를 실제 파일 ID로 쓰세요.
2. **어떻게 확인했나요?** 거리순 구성 사진과 K 변경 전후의 교차표에서 서로 다른 근거를 하나씩 찾으세요.
3. **무엇을 근거로 운영 방법을 정했나요?** 120장으로 만든 폴더와 새 사진 40장의 결과에서 수치를 하나씩 골라, 담당자가 자동 배정 결과를 어디까지 믿고 무엇을 검토할지 쓰세요.